# 🔥 PINN — Inverse Heat Source Identification (1D)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/pinn-inverse-heat/blob/main/notebooks/01_inverse_1D.ipynb)

---

## Problem Statement

Given the steady-state heat equation on $\Omega = [0,1]$:

$$-\frac{d^2 T}{dx^2} = f(x), \quad x \in (0,1)$$

$$T(0) = 0, \quad T(1) = 0$$

**Forward problem**: given $f(x)$, find $T(x)$.  
**Inverse problem** *(this notebook)*: given sparse noisy measurements $\{T(x_i^{obs})\}$, **recover $f(x)$**.

### Ground truth (for validation)

We choose:
$$f^*(x) = \sin(\pi x) + 0.5\sin(3\pi x)$$

whose analytical solution is:
$$T^*(x) = \frac{1}{\pi^2}\sin(\pi x) + \frac{0.5}{9\pi^2}\sin(3\pi x)$$

---

## Strategy

Two neural networks are trained **simultaneously**:

| Network | Input | Output | Role |
|---------|-------|--------|------|
| `PINN` $\mathcal{N}_T$ | $x$ | $T(x)$ | Satisfy PDE + BCs |
| `SourceNet` $\mathcal{N}_f$ | $x$ | $f(x)$ | Identify unknown source |

### Loss function

$$\mathcal{L} = \underbrace{w_{pde}\,\|\mathcal{N}_T'' + \mathcal{N}_f\|^2}_{\text{PDE residual}} + \underbrace{w_{bc}\,\mathcal{L}_{bc}}_{\text{BCs}} + \underbrace{w_{data}\,\|\mathcal{N}_T(x^{obs}) - T^{obs}\|^2}_{\text{data fit}} + \underbrace{w_{reg}\,\left\|\frac{d\mathcal{N}_f}{dx}\right\|^2}_{\text{Tikhonov reg.}}$$

In [ ]:
# ── Install if running on Colab ─────────────────────────────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install torch matplotlib numpy -q

In [ ]:
import sys, os
sys.path.append(os.path.join('..', 'src'))

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

torch.manual_seed(42)
np.random.seed(42)

print(f'PyTorch {torch.__version__}')

## 1. Analytical Ground Truth

In [ ]:
def f_true(x):
    """True source: f*(x) = sin(πx) + 0.5·sin(3πx)"""
    return np.sin(np.pi * x) + 0.5 * np.sin(3 * np.pi * x)

def T_true(x):
    """Analytical solution of -T'' = f*(x) with T(0)=T(1)=0"""
    return (1/np.pi**2) * np.sin(np.pi * x) + (0.5/(9*np.pi**2)) * np.sin(3 * np.pi * x)

x_plot = np.linspace(0, 1, 500)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x_plot, T_true(x_plot), 'b-', lw=2, label='$T^*(x)$')
axes[0].set_title('Temperature field $T^*(x)$', fontsize=13)
axes[0].set_xlabel('x'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(x_plot, f_true(x_plot), 'r-', lw=2, label='$f^*(x)$')
axes[1].set_title('Source term $f^*(x)$ — to be recovered', fontsize=13)
axes[1].set_xlabel('x'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Ground Truth', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/ground_truth.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Sparse Noisy Observations

In [ ]:
# ── Parameters ──────────────────────────────────────────────────────────────
N_OBS   = 15     # number of measurement points
NOISE   = 0.02   # relative noise level (σ = NOISE * max|T|)

# Observation points (avoid boundaries)
x_obs_np = np.linspace(0.05, 0.95, N_OBS)
T_clean  = T_true(x_obs_np)
T_obs_np = T_clean + NOISE * np.max(np.abs(T_clean)) * np.random.randn(N_OBS)

# Convert to tensors
x_obs = torch.tensor(x_obs_np, dtype=torch.float32).unsqueeze(1)
T_obs = torch.tensor(T_obs_np, dtype=torch.float32).unsqueeze(1)

# Collocation points
N_COLLOC = 200
x_colloc = torch.linspace(0, 1, N_COLLOC).unsqueeze(1)

print(f'Observations : {N_OBS} points  |  Noise level : {NOISE*100:.0f}%')
print(f'Collocation  : {N_COLLOC} points')

plt.figure(figsize=(8, 4))
plt.plot(x_plot, T_true(x_plot), 'b-', lw=2, alpha=0.5, label='$T^*$ (unknown)')
plt.scatter(x_obs_np, T_obs_np, c='red', s=60, zorder=5, label=f'Observations ({N_OBS} pts, noise={NOISE*100:.0f}%)')
plt.title('What the PINN sees: only sparse noisy measurements', fontsize=12)
plt.xlabel('x'); plt.legend(); plt.grid(alpha=0.3)
plt.savefig('../results/observations.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Model Architecture

In [ ]:
from model import PINN, SourceNetwork
from losses import total_loss

pinn       = PINN(layers=[1, 64, 64, 64, 1])
source_net = SourceNetwork(layers=[1, 32, 32, 32, 1])

n_pinn   = sum(p.numel() for p in pinn.parameters())
n_source = sum(p.numel() for p in source_net.parameters())
print(f'PINN parameters      : {n_pinn:,}')
print(f'SourceNet parameters : {n_source:,}')
print(f'Total                : {n_pinn + n_source:,}')

## 4. Training

In [ ]:
import torch.optim as optim

# ── Hyperparameters ──────────────────────────────────────────────────────────
W_PDE   = 1.0
W_BC    = 100.0
W_DATA  = 100.0
W_REG   = 1e-3     # Tikhonov weight — increase to enforce smoother f
REG_ORD = 1        # 1st-order Tikhonov (penalises |df/dx|²)

ADAM_EPOCHS  = 5000
LBFGS_STEPS  = 300
SNAP_EVERY   = 200

# ── Storage ──────────────────────────────────────────────────────────────────
history   = {'loss': [], 'pde': [], 'bc': [], 'data': [], 'reg': []}
snapshots = []  # (epoch_label, x_arr, T_arr, f_arr)

x_eval = torch.linspace(0, 1, 400).unsqueeze(1)

all_params     = list(pinn.parameters()) + list(source_net.parameters())
optimizer_adam = optim.Adam(all_params, lr=1e-3)

# ── Phase 1 : Adam ───────────────────────────────────────────────────────────
print('Phase 1 — Adam')
for epoch in range(1, ADAM_EPOCHS + 1):
    optimizer_adam.zero_grad()
    L = total_loss(pinn, source_net, x_colloc, x_obs, T_obs,
                   w_pde=W_PDE, w_bc=W_BC, w_data=W_DATA,
                   w_reg=W_REG, reg_order=REG_ORD)
    L['total'].backward()
    optimizer_adam.step()

    history['loss'].append(L['total'].item())
    history['pde'].append(L['pde'])
    history['bc'].append(L['bc'])
    history['data'].append(L['data'])
    history['reg'].append(L['reg'])

    if epoch % SNAP_EVERY == 0 or epoch == 1:
        with torch.no_grad():
            T_s = pinn(x_eval).squeeze().numpy()
            f_s = source_net(x_eval).squeeze().numpy()
        snapshots.append((epoch, x_eval.squeeze().numpy(), T_s, f_s))

    if epoch % 1000 == 0:
        print(f'  {epoch:5d} | total {L["total"]:.3e} | '
              f'pde {L["pde"]:.2e} | data {L["data"]:.2e}')

# ── Phase 2 : L-BFGS ─────────────────────────────────────────────────────────
print('\nPhase 2 — L-BFGS')
optimizer_lbfgs = optim.LBFGS(all_params, lr=1.0, max_iter=20,
                               history_size=50, line_search_fn='strong_wolfe')
lbfgs_log = []

def closure():
    optimizer_lbfgs.zero_grad()
    L = total_loss(pinn, source_net, x_colloc, x_obs, T_obs,
                   w_pde=W_PDE, w_bc=W_BC, w_data=W_DATA,
                   w_reg=W_REG, reg_order=REG_ORD)
    L['total'].backward()
    lbfgs_log.append(L['total'].item())
    return L['total']

for step in range(LBFGS_STEPS):
    optimizer_lbfgs.step(closure)
    if (step + 1) % 100 == 0:
        print(f'  step {step+1:4d} | loss {lbfgs_log[-1]:.4e}')

# Final snapshot
with torch.no_grad():
    T_final = pinn(x_eval).squeeze().numpy()
    f_final = source_net(x_eval).squeeze().numpy()
snapshots.append(('final', x_eval.squeeze().numpy(), T_final, f_final))
print('\nTraining complete ✓')

## 5. Results — Comparison with Analytical Solution

In [ ]:
x_np = x_eval.squeeze().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Temperature ──
ax = axes[0]
ax.plot(x_plot, T_true(x_plot), 'b-', lw=2.5, label='Analytical $T^*$')
ax.plot(x_np, T_final, 'r--', lw=2, label='PINN prediction')
ax.scatter(x_obs_np, T_obs_np, c='gray', s=50, zorder=5, alpha=0.7, label='Observations')
ax.set_title('Temperature field $T(x)$', fontsize=13)
ax.set_xlabel('x'); ax.legend(); ax.grid(alpha=0.3)

T_err = np.abs(T_final - T_true(x_np))
ax.text(0.05, 0.95, f'Max error: {T_err.max():.2e}\nL2 error: {np.mean(T_err**2)**0.5:.2e}',
        transform=ax.transAxes, va='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8), fontsize=10)

# ── Source term ──
ax = axes[1]
ax.plot(x_plot, f_true(x_plot), 'b-', lw=2.5, label='True $f^*(x)$')
ax.plot(x_np, f_final, 'r--', lw=2, label='PINN recovered $f$')
ax.set_title('Source term $f(x)$ — recovered by PINN', fontsize=13)
ax.set_xlabel('x'); ax.legend(); ax.grid(alpha=0.3)

f_err = np.abs(f_final - f_true(x_np))
ax.text(0.05, 0.95, f'Max error: {f_err.max():.2e}\nL2 error: {np.mean(f_err**2)**0.5:.2e}',
        transform=ax.transAxes, va='top',
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8), fontsize=10)

plt.suptitle('PINN Inverse Problem — Final Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/final_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Loss History

In [ ]:
epochs = np.arange(1, len(history['loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].semilogy(epochs, history['loss'], 'k-', lw=1.5, label='Total loss')
axes[0].set_title('Total loss (Adam phase)', fontsize=12)
axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].semilogy(epochs, history['pde'],  label='PDE residual')
axes[1].semilogy(epochs, history['bc'],   label='BC loss')
axes[1].semilogy(epochs, history['data'], label='Data loss')
axes[1].semilogy(epochs, history['reg'],  label='Tikhonov reg')
axes[1].set_title('Loss decomposition', fontsize=12)
axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../results/loss_history.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Training Animation (GIF)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

def animate(i):
    epoch_label, x_s, T_s, f_s = snapshots[i]
    for ax in axes:
        ax.cla()

    # Temperature
    axes[0].plot(x_plot, T_true(x_plot), 'b-', lw=2, alpha=0.6, label='$T^*$ (true)')
    axes[0].plot(x_s, T_s, 'r-', lw=2, label='PINN')
    axes[0].scatter(x_obs_np, T_obs_np, c='gray', s=40, zorder=5, alpha=0.7)
    axes[0].set_ylim(-0.005, T_true(x_plot).max() * 1.3)
    axes[0].set_title(f'Temperature  —  epoch {epoch_label}', fontsize=12)
    axes[0].set_xlabel('x'); axes[0].legend(loc='upper right'); axes[0].grid(alpha=0.3)

    # Source
    axes[1].plot(x_plot, f_true(x_plot), 'b-', lw=2, alpha=0.6, label='$f^*$ (true)')
    axes[1].plot(x_s, f_s, 'r-', lw=2, label='PINN')
    axes[1].set_ylim(f_true(x_plot).min() * 1.5, f_true(x_plot).max() * 1.5)
    axes[1].set_title(f'Recovered source $f(x)$  —  epoch {epoch_label}', fontsize=12)
    axes[1].set_xlabel('x'); axes[1].legend(loc='upper right'); axes[1].grid(alpha=0.3)

    plt.tight_layout()

ani = animation.FuncAnimation(fig, animate, frames=len(snapshots), interval=200, repeat=True)
ani.save('../results/training_animation.gif', writer='pillow', fps=5, dpi=100)
plt.close()
print('GIF saved → results/training_animation.gif')

# Display inline
from IPython.display import Image
Image('../results/training_animation.gif')

## 8. Noise Sensitivity Analysis

In [ ]:
noise_levels = [0.0, 0.01, 0.02, 0.05, 0.10]
results_noise = {}

for noise in noise_levels:
    torch.manual_seed(0); np.random.seed(0)

    T_noisy = T_clean + noise * np.max(np.abs(T_clean)) * np.random.randn(N_OBS)
    x_obs_t = torch.tensor(x_obs_np, dtype=torch.float32).unsqueeze(1)
    T_obs_t = torch.tensor(T_noisy,  dtype=torch.float32).unsqueeze(1)

    _pinn   = PINN(layers=[1, 64, 64, 64, 1])
    _source = SourceNetwork(layers=[1, 32, 32, 32, 1])
    _params = list(_pinn.parameters()) + list(_source.parameters())
    _opt    = optim.Adam(_params, lr=1e-3)

    for _ in range(3000):
        _opt.zero_grad()
        L = total_loss(_pinn, _source, x_colloc, x_obs_t, T_obs_t,
                       w_reg=W_REG, reg_order=REG_ORD)
        L['total'].backward()
        _opt.step()

    with torch.no_grad():
        f_pred = _source(x_eval).squeeze().numpy()

    f_l2 = np.mean((f_pred - f_true(x_np))**2)**0.5
    results_noise[noise] = (f_pred, f_l2)
    print(f'  noise={noise*100:4.1f}%  →  f L2 error = {f_l2:.4f}')

# ── Plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cmap = plt.cm.viridis
colors = [cmap(i / len(noise_levels)) for i in range(len(noise_levels))]

axes[0].plot(x_plot, f_true(x_plot), 'k-', lw=3, label='$f^*$ (true)', zorder=10)
for (noise, (f_pred, _)), color in zip(results_noise.items(), colors):
    axes[0].plot(x_np, f_pred, '--', lw=1.5, color=color, label=f'noise={noise*100:.0f}%')
axes[0].set_title('Effect of noise on source recovery', fontsize=12)
axes[0].set_xlabel('x'); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

nv = [n * 100 for n in noise_levels]
l2s = [results_noise[n][1] for n in noise_levels]
axes[1].plot(nv, l2s, 'ro-', lw=2, ms=8)
axes[1].set_title('L² error vs noise level', fontsize=12)
axes[1].set_xlabel('Noise level (%)'); axes[1].set_ylabel('L² error on $f$')
axes[1].grid(alpha=0.3)

plt.suptitle('Noise Sensitivity Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/noise_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Tikhonov Regularization Study

In [ ]:
reg_weights = [0.0, 1e-5, 1e-3, 1e-1]
results_reg = {}

for w_reg in reg_weights:
    torch.manual_seed(0); np.random.seed(0)
    _pinn   = PINN(layers=[1, 64, 64, 64, 1])
    _source = SourceNetwork(layers=[1, 32, 32, 32, 1])
    _params = list(_pinn.parameters()) + list(_source.parameters())
    _opt    = optim.Adam(_params, lr=1e-3)

    x_obs_t = torch.tensor(x_obs_np, dtype=torch.float32).unsqueeze(1)
    T_noisy = T_clean + 0.03 * np.max(np.abs(T_clean)) * np.random.randn(N_OBS)
    T_obs_t = torch.tensor(T_noisy, dtype=torch.float32).unsqueeze(1)

    for _ in range(3000):
        _opt.zero_grad()
        L = total_loss(_pinn, _source, x_colloc, x_obs_t, T_obs_t, w_reg=w_reg)
        L['total'].backward()
        _opt.step()

    with torch.no_grad():
        f_pred = _source(x_eval).squeeze().numpy()
    f_l2 = np.mean((f_pred - f_true(x_np))**2)**0.5
    results_reg[w_reg] = (f_pred, f_l2)
    print(f'  w_reg={w_reg:.0e}  →  f L2 error = {f_l2:.4f}')

plt.figure(figsize=(9, 5))
plt.plot(x_plot, f_true(x_plot), 'k-', lw=3, label='$f^*$ true')
cmap2 = plt.cm.plasma
for i, (wr, (fp, l2)) in enumerate(results_reg.items()):
    plt.plot(x_np, fp, '--', lw=1.8, color=cmap2(i/len(reg_weights)),
             label=f'$w_{{reg}}={wr:.0e}$ (L2={l2:.3f})')
plt.title('Effect of Tikhonov Regularization Weight (noise=3%)', fontsize=12)
plt.xlabel('x'); plt.legend(fontsize=10); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../results/tikhonov_study.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

| Feature | Result |
|---------|--------|
| Temperature L² error | See Section 5 |
| Source L² error | See Section 5 |
| Noise robustness | ✅ Good up to ~5%, degrades at 10% |
| Tikhonov effect | ✅ Prevents overfitting noisy data |
| Training time | ~2 min (CPU) |

### Key takeaways
1. **Two-network architecture** allows simultaneous field reconstruction and source identification.
2. **Tikhonov regularization** is critical for stable inversion — without it, $f$ overfits noise.
3. **L-BFGS** after Adam dramatically improves convergence quality.
4. The method is robust to moderate noise (~2–5%) with only 15 measurement points.

### Next steps
- 📐 Extend to 2D: `02_inverse_2D.ipynb`
- 🔁 Try time-dependent source: $-\partial_t T + \partial_{xx} T = f(x,t)$
- 📊 Compare VPINN formulation for higher-order accuracy